In [31]:
from google.colab import drive
drive.mount('/content/drive')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [37]:
import os
import sys
from pathlib import Path


PROJECT_PATH = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

if str(PROJECT_PATH) not in sys.path:
    sys.path.insert(0, str(PROJECT_PATH))

print("Project path:", PROJECT_PATH)

Project path: /content


In [38]:
PROJECT_DIR = Path("/content/drive/MyDrive/rag-chatbot-evaluation-framework")
SRC_PATH = PROJECT_DIR / "src"

print("PROJECT_PATH exists:", PROJECT_DIR.exists())
print("SRC_PATH exists:", SRC_PATH.exists())
print("Files in src:", os.listdir(SRC_PATH))

if str(SRC_PATH) not in sys.path:
    sys.path.insert(0, str(SRC_PATH))

from document_loader import load_markdown_documents, validate_documents
from text_splitter import create_chunks
from vector_store import FaissVectorStore
from rag_pipeline import SimpleRAGPipeline

DOCUMENT_DIR = PROJECT_DIR / "data" / "documents"

docs = load_markdown_documents(DOCUMENT_DIR)
validate_documents(docs)

chunks = create_chunks(docs, chunk_size=500, overlap=100)
vector_store = FaissVectorStore()
vector_store.build_index(chunks)

print('Vector index built successfully.')
print('total chunks:', len(chunks))

PROJECT_PATH exists: True
SRC_PATH exists: True
Files in src: ['__init__.py', 'rag_pipeline.py', 'text_splitter.py', 'document_loader.py', 'vector_store.py', '__pycache__', 'evaluator.py', 'report_generator.py']
Document validation passed. Loaded 5 documents.


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Vector index built successfully.
total chunks: 10


In [43]:
query = "How long standard shipping usually take?"

retrival_results = vector_store.search(query, top_k=3)

for result in retrival_results:
  print('Source', result['source'])
  print('Score', round(result['similarity_score'], 4))
  print(result['text'][:300])
  print('-' * 80)

Source shipping_policy.md
Score 0.7538
# Shipping Policy

Standard shipping usually takes 3 to 5 business days.

Express shipping usually takes 1 to 2 business days.

Orders above $50 qualify for free standard shipping.

Shipping delays may occur during public holidays, extreme weather conditions, or periods of high order volume.

Custom
--------------------------------------------------------------------------------
Source return_policy.md
Score 0.4188
# Return Policy

Customers can return most products within 30 days of delivery.

Returned items must be unused, undamaged, and in their original packaging.

Refunds are usually processed within 5 to 10 business days after the returned item is received and inspected.

Final sale items, gift cards, an
--------------------------------------------------------------------------------
Source shipping_policy.md
Score 0.3132
rder has not arrived after the estimated delivery date, customers should contact customer support for assistance.
-------

In [44]:
rag = SimpleRAGPipeline(vector_store, top_k=3)
questions = [
    'How long standard shipping usually take?',
    'Can customized products be returned?',
    'What payment method does the company accept?',
    'Can customer support see my password?'
]

for question in questions:
  response = rag.ask(question)
  print('Question:', question)
  print('Answer:', response['answer'])
  print('Source:', response['sources'])
  print('=' * 100)

Question: How long standard shipping usually take?
Answer: # Shipping Policy

Standard shipping usually takes 3 to 5 business days. Express shipping usually takes 1 to 2 business days. Source: shipping_policy.md.
Source: ['shipping_policy.md', 'return_policy.md', 'shipping_policy.md']
Question: Can customized products be returned?
Answer: Final sale items, gift cards, and customized products cannot be returned. # Return Policy

Customers can return most products within 30 days of delivery. Source: return_policy.md.
Source: ['return_policy.md', 'warranty_policy.md', 'return_policy.md']
Question: What payment method does the company accept?
Answer: # Payment Policy

The company accepts credit cards, debit cards, PayPal, and selected digital wallets. If payment fails, the order will not be processed until a valid payment method is provided. Source: payment_policy.md.
Source: ['payment_policy.md', 'payment_policy.md', 'warranty_policy.md']
Question: Can customer support see my password?
An